In [53]:
import json
from pathlib import Path
import pandas as pd
from data import load_data

In [54]:
data_path = r"D:\Python Projects\Hospital readmission risk\data\raw\diagnoses_and_following.csv"
dictionary_path = r"D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\diagnoses_dictionary.csv"
STATE_PATH = Path("D:\Python Projects\Hospital readmission risk\data\intermediate\diagnosess_snomed_state.json")
concept_path = r"D:\Python Projects\Hospital readmission risk\data\concepts"
write_path = r"D:\Python Projects\Hospital readmission risk\data\processed\related_diagnoses.csv"
CACHE: dict[str, str] = {}
ANCESTORS: dict[str, set[str]] = {}
"""
any null diagnoses will be 0
"""
sql = """
with sec_codes as (
  select 
  stay_id,
  following_stay_id as fol_id,
  readmit_30d,
  readmit_90d,
  main.main_diagnosis_code sec_code
  from `hospital-readmission-4.helper_tables.helper_utilization` hu
  left join `hospital-readmission-4.data_slim.main_diagnoses` main
  on hu.following_stay_id = main.id
)
SELECT
  sec.stay_id,
  main.main_diagnosis_code as code,
  sec.fol_id, 
  sec.sec_code,
  readmit_30d,
  readmit_90d
FROM sec_codes sec
left join `hospital-readmission-4.data_slim.main_diagnoses` main
on main.id = sec.stay_id
"""
output_cols = ['is_related']
basic_set = {
  '138875005', #snomed concept
  '404684003', #clinical finding
  '64572001', #disease(disorder)
  '844005', #behaviour finding
  '362965005', #disorder of body system
  '417163006', #injury
  '76712006' #disorder of digestive organ
  '118228005', #functional finding
  '27624003', #chronic disease
  '105612003', #Injury of internal organ
  '302768007', #Employment finding
  '384821006', #mental state bs
  '365526009', #job details finding
  '34014006', #viral disease, could be different viruses
  '128139000', # Inflammatory disorder
  '22253000', #Pain
   '118228005', #Finding by function
   '302292003', #finding of trunk structure
   '363171009', #Inflammation of specific body systems
  }

In [55]:
def load_state():

    global CACHE

    if not STATE_PATH.exists():
        return

    with STATE_PATH.open("r", encoding="utf-8") as f:
        data = json.load(f)

    raw_cache = data.get("cache", {})
    CACHE = {code: path for code, path in raw_cache.items()}

In [56]:
def read_concept(concept_path):

    path = Path(concept_path)

    if not path.exists():
        return {}
    
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

In [57]:
def get_concept(concept_id: str) -> dict:

    if concept_id in CACHE:

        return read_concept(CACHE[concept_id])

In [58]:
def get_parent_ids(concept: dict) -> list[str]:

    """Extract parent conceptIds from the concept JSON (adjust to actual structure)."""

    parents = []

    for rel in concept.get("relationships", []):
        # 'is a' relationship has conceptId 116680003 in SNOMED CT
        if rel.get("type", {}).get("conceptId") == "116680003" and rel.get("active"):

            target = rel.get("target") or {}

            cid = target.get("conceptId")

            if cid:
                
                parents.append(cid)
            
    return parents

In [59]:
def is_or_has_ancestor_in(concept_id: str, target_ids: set[str], max_depth: int = 10) -> bool:
    # if we’ve already fully explored this concept’s ancestors in a previous run,
    # just return the previous answer if you also store it, or skip re-walk.
    visited = set()
    frontier = {concept_id}

    if concept_id in target_ids:
        return 1, {concept_id}

    for depth in range(max_depth):

        next_frontier = set()

        for cid in frontier:

            if cid in visited:
                continue

            visited.add(cid)

            c = get_concept(cid)

            if not isinstance(c, dict):
                continue

            parents = get_parent_ids(c)

            if target_ids.intersection(parents):
                return 1, target_ids.intersection(parents)
            
            next_frontier.update(p for p in parents if p not in visited)

        if not next_frontier:
            break
        
        frontier = next_frontier

    return -1, {}

In [60]:
def find_all_ancestors(concept_id: str, basic_set: set):

    if concept_id in ANCESTORS:
        return ANCESTORS[concept_id]

    ancestors : set[str] = set()
    frontier = {concept_id}
    visited = set()

    for depth in range(10):

        new_frontier = set()

        for cid in frontier:

            if cid in visited or cid in basic_set:
                continue
            
            c = get_concept(cid)

            if c is {}:
                continue

            parents = get_parent_ids(c)

            for parent in parents:

                if parent in basic_set:
                    continue

                ancestors.add(parent)
                new_frontier.add(parent)
            
        if not new_frontier:
            break
        
        frontier = new_frontier

    ANCESTORS[concept_id] = ancestors
            
    return ancestors

In [61]:
def get_relation(concept_id1, concept_id2, basic_set: set):

    if pd.isna(concept_id1) or pd.isna(concept_id2):

        return 0, {}

    ancestors1 = find_all_ancestors(str(int(concept_id1)), basic_set)
    
    return is_or_has_ancestor_in(str(int(concept_id2)), ancestors1)

In [62]:
def pack_dictionary(data, path: str):

    data.to_csv(path)

In [63]:
def get_description(concept_id: str):

    if CACHE[concept_id]:

        return get_concept(concept_id)["descriptions"][0]["term"]

    return None

In [68]:
def build_relations(data):

    data.set_index('stay_id', inplace = True)
    
    rows = []
    for row in data.itertuples():

        is_related, relation_list = get_relation(row.code, row.sec_code, basic_set)
        
        rows.append({
            'stay_id': row.Index,
            'is_related' : is_related,
            #'relation_list' : str(relation_list)
        })

    relations = pd.DataFrame(rows).set_index('stay_id')

    flag_cols = ['readmit_30d', 'readmit_90d']

    relations[flag_cols] = data[flag_cols]

    relations['rel_readmit_30d'] = (relations['readmit_30d'] * relations['is_related']).clip(lower = 0)
    relations['rel_readmit_90d'] = (relations['readmit_90d'] * relations['is_related']).clip(lower = 0)

    return relations    

In [ ]:
if __name__ == '__main__':

    data = load_data(data_path, sql)
    load_state()
    relations = build_relations(data)
    #pack_dictionary(relations, write_path)

In [72]:
pack_dictionary(relations, write_path)

---

In [70]:
relations[relations['rel_readmit_30d'] == 1]

,is_related,readmit_30d,readmit_90d,rel_readmit_30d,rel_readmit_90d
stay_id,,,,,
063ec749-39a0-076f-67f0-5e5837e7de33,1,1,1,1,1
0790b98a-93a4-7273-2e65-a9bbcda7d22e,1,1,1,1,1
1ac48e53-1d93-e30d-be12-1e819714012d,1,1,1,1,1
25630705-6421-63dd-90f0-71f015da08c2,1,1,1,1,1
3211276c-c59a-a786-8d82-f858a9810cd0,1,1,1,1,1
...,...,...,...,...,...
ffa10dfd-b62d-a197-5f89-39944f00094b,1,1,1,1,1
ffa10dfd-b62d-a197-b551-e78a7f4d3848,1,1,1,1,1
ffa10dfd-b62d-a197-1623-fd9959551de6,1,1,1,1,1


In [48]:
data.loc['a00cc7a8-db17-190a-d817-17760b981851']

code                                    307426000.0
fol_id         a00cc7a8-db17-190a-fdc5-f68bcfbfac8e
sec_code                                363406005.0
readmit_30d                                       0
readmit_90d                                       0
Name: a00cc7a8-db17-190a-d817-17760b981851, dtype: object

In [14]:
import ast
import pandas as pd
interactions = pd.Series(relations['relation_list'].unique())
interactions = interactions.apply(ast.literal_eval)
interactions

0                                              {}
1     {239999004, 263139003, 1290488009, 6849006}
2                                      {76712006}
3                                     {444158007}
4                                     {118228005}
                         ...                     
85                                     {57054005}
86                                     {34014006}
87                                     {19829001}
88                                    {384821006}
89                         {124042003, 238037008}
Length: 90, dtype: object

In [16]:
codes = set()
for el in interactions:
    codes.update(el)

intersects = []

for el in codes:

    intersects.append(
        {'code': el,
        'description' : get_description(el)
        })

print(pd.DataFrame(intersects).set_index('code'))

,description
code,
81308009,Disease of brain
44241007,Heart valve stenosis
301358001,Pain of cardiovascular structure
105612003,Injury of internal organ
302768007,Employment finding
...,...
22253000,Pain
406123005,Viscus structure finding (finding)
118938008,Disease of mouth


In [52]:
pd.set_option('display.max_rows', 100) 
print(pd.DataFrame(intersects).set_index('code').iloc[77:104])

                                                  description
code                                                         
118228005                                 Finding by function
363169009                Inflammation of specific body organs
700012005              Disorder of free lower limb (disorder)
128482007                         Acute inflammatory disorder
1290488009                         Disorder of tendon of knee
126926005                                  Neoplasm of breast
125603006                                     Injury of ankle
301226008                     Lower respiratory tract finding
239999004                   Soft tissue lesion of knee region
128134005                             Fallopian tube disorder
128292002           Chronic disorder of cardiovascular system
102957003                                Neurological finding
60492000                                 Disorder of ligament
105606008                    Injury of musculoskeletal system
12812100